# 🌳🌲 Урок 9 — Случайный лес: много деревьев умнее одного

**Что мы сделаем сегодня:**
1. Обучим одно дерево и увидим, как оно «зазубривает» данные (переобучение).
2. Соберём из ста разных деревьев **случайный лес** и сравним точность.
3. Устроим турнир: KNN против Дерева против Леса.
4. Узнаем, **на что смотрит** модель (важность признаков).
5. Проверим, сколько деревьев реально нужно.

> 💡 **Главная идея:** одно дерево — как один эксперт, может ошибиться уверенно. Лес — как класс, который голосует: случайные ошибки отдельных деревьев гасят друг друга.

*Работаем на знакомом датасете Titanic (тот же, что на уроке 7).*

## Блок 1 · Загружаем данные и быстро готовим их

**Что делаем:** берём датасет Titanic, выбираем понятные признаки и приводим их к числам.

**Зачем:** модель понимает только числа, а пол записан текстом; ещё есть пропуски в возрасте.

**Что ожидаем:** таблицу из 6 признаков без пропусков.

> ⚠️ Здесь мы готовим данные вручную. Это **последний урок**, где мы так делаем — на уроке 10 всю подготовку возьмёт на себя Pipeline.

In [ ]:
import seaborn as sns
import pandas as pd

df = sns.load_dataset('titanic')

# Признаки, которые используем; target — survived (1 = выжил, 0 = нет)
cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']
data = df[cols + ['survived']].copy()

# Пропуски в возрасте заполняем медианой
data['age'] = data['age'].fillna(data['age'].median())

# Пол текстом -> числом: male=0, female=1
data['sex'] = (data['sex'] == 'female').astype(int)

X = data[cols]        # признаки
y = data['survived']  # ответ, который предсказываем
X.head()

## Блок 2 · Baseline: одно дерево. Ловим переобучение

**Что делаем:** обучаем одно дерево решений и сравниваем точность на train и на test.

**Зачем:** чтобы увидеть проблему одного дерева — оно «списывает с ответов» на train.

**Что ожидаем:** на train почти 100%, на test заметно ниже. Это и есть переобучение.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# random_state=42 — одинаковое разбиение у всех в классе (можно сравнивать)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

# Проверка на train = 'списать с ответов'. Проверка на test — честная.
acc_train = accuracy_score(y_train, tree.predict(X_train))
acc_test  = accuracy_score(y_test,  tree.predict(X_test))
print(f'Дерево  train: {acc_train:.1%}   test: {acc_test:.1%}')

**Вывод:** видите разрыв? На train дерево знает почти всё, на test ошибается заметно чаще. Оно запомнило конкретных пассажиров, а не общее правило. Починим это лесом.

## Блок 3 · Случайный лес: 100 деревьев голосуют

**Что делаем:** сажаем 100 деревьев. Каждое учится на своей случайной части данных и признаков.

**Зачем:** разные деревья ошибаются по-разному, и голосование сглаживает случайные ошибки.

**Что ожидаем:** точность на test **выше**, чем у одного дерева.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# n_estimators=100 — сто разных деревьев
forest = RandomForestClassifier(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)

acc_forest = accuracy_score(y_test, forest.predict(X_test))
print(f'Одно дерево, test: {acc_test:.1%}')
print(f'Лес (100),   test: {acc_forest:.1%}')

**Вывод:** лес обычно обыгрывает одно дерево. Мы не придумали новый умный алгоритм — мы просто заставили много обычных деревьев **проголосовать**.

## Блок 4 · Турнир: KNN vs Дерево vs Лес

**Что делаем:** обучаем три модели на одних и тех же данных и сравниваем на одном test.

**Зачем:** честное сравнение — единственный способ понять, кто сильнее.

**Что ожидаем:** обычно лидирует лес. KNN может проседать — он чувствителен к масштабу признаков (это станет важным на уроке 10).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

models = {
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Дерево':    DecisionTreeClassifier(random_state=42),
    'Лес (100)': RandomForestClassifier(n_estimators=100, random_state=42),
}

print('Модель            Accuracy на test')
print('-' * 34)
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f'{name:16s} {acc:.1%}')

## Блок 5 · На что смотрит лес? Важность признаков

**Что делаем:** спрашиваем у леса, какие признаки он использовал чаще и полезнее всего.

**Зачем:** понять логику модели своими глазами.

**Что ожидаем:** обычно на первом месте пол и класс каюты.

> ❗ **Важно:** высокая важность признака — это НЕ доказательство причины. Это лишь «модель часто этим пользовалась». Помним урок про корреляцию и причинность!

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(forest.feature_importances_, index=X.columns)
importances = importances.sort_values()

importances.plot(kind='barh', color='#5B4FC4')
plt.title('Важность признаков (Random Forest)')
plt.xlabel('вклад признака')
plt.tight_layout()
plt.show()

## Блок 6 (эксперимент) · Сколько деревьев реально нужно?

**Что делаем:** обучаем лес с разным числом деревьев и смотрим на точность.

**Зачем:** проверить миф «больше деревьев = всегда лучше».

**Что ожидаем:** сначала точность растёт, потом выходит на **плато**.

In [ ]:
ns = [1, 5, 10, 50, 100, 300]
accs = []
for n in ns:
    m = RandomForestClassifier(n_estimators=n, random_state=42)
    m.fit(X_train, y_train)
    accs.append(accuracy_score(y_test, m.predict(X_test)))

plt.plot(ns, accs, marker='o', color='#0E7C86')
plt.title('Точность vs число деревьев')
plt.xlabel('n_estimators'); plt.ylabel('accuracy на test')
plt.grid(True, alpha=0.3)
plt.show()

**Вывод:** больше деревьев ≠ бесконечно лучше. После некоторого числа прирост почти останавливается, а считать дольше. Деревья дают **стабильность**, а не бесконечный рост.

---
## 🏠 Домашнее задание
1. Возьми свой датасет (или Titanic) и обучи Random Forest.
2. Выведи **топ-5** важных признаков.
3. Письменно (3–4 предложения): какой признак самый важный и логично ли это по смыслу задачи?
4. Найди число деревьев, после которого точность почти не растёт.

**Сдача:** ссылка на этот ноутбук в Telegram.

---
## ⭐ Для тех, кто справился быстрее
Запусти ячейку ниже: `oob_score` — «бесплатная» проверка на примерах, которые дерево не видело.

In [ ]:
# OOB (out-of-bag): лес сам себя проверяет на не попавших в выборку примерах
forest_oob = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=42)
forest_oob.fit(X_train, y_train)
print(f'OOB score:  {forest_oob.oob_score_:.1%}')
print(f'Test score: {accuracy_score(y_test, forest_oob.predict(X_test)):.1%}')
# Сравни два числа: OOB — честная оценка без отдельного теста

In [ ]:
# Подбор параметров через GridSearchCV (перебор + проверка)
from sklearn.model_selection import GridSearchCV

grid = {'n_estimators': [50, 100, 200], 'max_depth': [4, 6, 8, None]}
search = GridSearchCV(RandomForestClassifier(random_state=42), grid, cv=5)
search.fit(X_train, y_train)
print('Лучшие параметры:', search.best_params_)
print(f'Лучшая точность (CV): {search.best_score_:.1%}')